In [ ]:
%pip install pytest -q

# INT-01 — End-to-End Integration Tests

**Ticket:** INT-01  
**Purpose:** Validate the full Bronze → Silver → Gold → Dashboard pipeline.

Runs **123 pytest tests** across 4 test modules:

| Module | Coverage |
|--------|----------|
| `test_int01_bronze.py` | Table exists, schema, min row count |
| `test_int01_silver.py` | I-03 structure, I-04 rules, I-05 derived, I-06 referential integrity |
| `test_int01_gold.py` | Fact schema, grain uniqueness, metrics, dims, dashboard readiness |
| `test_int01_pipeline.py` | Cross-layer row count & revenue reconciliation |

> **Note:** Tests must run in-process via `pytest.main()` so the active Spark Connect session is available.

---

In [0]:
import sys
import os

# Prevent __pycache__ writes in workspace directories
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

# Derive project root from the notebook's workspace path.
# os.getcwd() returns /databricks/driver in Databricks jobs, not the project
# directory, so we use dbutils to get the actual workspace notebook path.
# Notebook lives at: /Workspace/.../tests/run_int01_tests
# Project root is two levels up: /Workspace/...
_nb_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
project_root = "/Workspace" + "/".join(_nb_path.split("/")[:-2])
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

# Force-reload src modules to pick up latest code
import importlib  # noqa: E402
import src.constants  # noqa: E402
import src.transforms  # noqa: E402
import src.validators  # noqa: E402

importlib.reload(src.constants)
importlib.reload(src.transforms)
importlib.reload(src.validators)

import pytest  # noqa: E402

exit_code = pytest.main(
    [
        "tests/test_int01_bronze.py",
        "tests/test_int01_silver.py",
        "tests/test_int01_gold.py",
        "tests/test_int01_pipeline.py",
        "-v",
        "--tb=short",
        "-p",
        "no:cacheprovider",
    ]
)

# Fail the notebook if any tests failed
assert exit_code == 0, f"INT-01 tests failed with exit code {exit_code}"